<a href="https://colab.research.google.com/github/jeetgoyal80/Bus-booking-app/blob/main/Smart_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p /content/smart_product_pricing/dataset
!mkdir -p /content/smart_product_pricing/artifacts
!mkdir -p /content/smart_product_pricing/outputs


In [2]:
!pip install -q pandas numpy scikit-learn lightgbm sentence-transformers timm torchvision pillow requests tqdm

In [ ]:
!python colab_price_pipeline.py

python3: can't open file '/content/colab_price_pipeline.py': [Errno 2] No such file or directory


In [3]:
import os
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

# ML
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

# Embeddings
from sentence_transformers import SentenceTransformer
import torch
import timm
from torchvision import transforms
from PIL import Image
import requests


In [4]:
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = s.replace('\n', ' ').replace('\r', ' ')
    s = re.sub(r'http\S+', ' ', s)
    s = re.sub(r'[^A-Za-z0-9.,%\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.lower()

def make_catalog_text(row):
    if 'catalog_content' in row and pd.notna(row['catalog_content']):
        return str(row['catalog_content'])
    parts = []
    for c in ['Item Name','Product Description','Bullet Point 1','Bullet Point 2','Bullet Point 3']:
        if c in row and pd.notna(row[c]):
            parts.append(str(row[c]))
    if 'Value' in row and pd.notna(row['Value']):
        parts.append(f"value:{row['Value']}")
    if 'Unit' in row and pd.notna(row['Unit']):
        parts.append(f"unit:{row['Unit']}")
    return " ".join(parts)

def extract_ipq_from_text(s):
    if pd.isna(s) or not s:
        return 1.0
    s = str(s).lower()
    m = re.search(r'pack of (\d+)', s)
    if m:
        return float(m.group(1))
    m = re.search(r'(\d+)\s*[x×]\s*\d+', s)
    if m:
        return float(m.group(1))
    return 1.0

def download_image(url, dest_path, timeout=8):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(url, timeout=timeout, headers=headers)
        if r.status_code == 200:
            with open(dest_path, 'wb') as f:
                f.write(r.content)
            return True
    except Exception:
        return False
    return False


In [5]:
DATA_DIR = '/content/smart_product_pricing/dataset'
ARTIFACTS = '/content/smart_product_pricing/artifacts'
os.makedirs(ARTIFACTS, exist_ok=True)

TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV  = os.path.join(DATA_DIR, 'test.csv')

TEXT_MODEL = 'all-MiniLM-L6-v2'
IMAGE_MODEL = 'resnet50'
BATCH_SIZE = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)


Device: cuda


In [6]:
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
print('train', train.shape, 'test', test.shape)

train (75000, 4) test (75000, 3)


In [7]:
for df in (train, test):
    # Combine catalog fields into a single text column
    df['catalog_content'] = df.apply(make_catalog_text, axis=1)

    # Clean text
    df['clean_text'] = df['catalog_content'].map(clean_text)

    # Text features
    df['num_words'] = df['clean_text'].str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)
    df['num_chars'] = df['clean_text'].str.len().fillna(0)

    # Image availability
    df['has_image'] = df['image_link'].notna().astype(int) if 'image_link' in df.columns else 0

    # Numeric value extraction logic
    if 'Value' in df.columns:
        df['value'] = pd.to_numeric(df['Value'], errors='coerce').fillna(0.0)
    else:
        # Extract all numbers per row (some rows might have none)
        extracted = df['clean_text'].str.extractall(r'(\d+(?:\.\d+)?)')[0]

        # Safely join multiple matches per row, and handle missing rows
        extracted_grouped = extracted.groupby(level=0).apply(lambda x: ' '.join(x)).reindex(df.index, fill_value='')

        # Take the first numeric token if multiple exist
        df['value'] = extracted_grouped.str.split().str[0]

        # Convert to float safely (invalid strings → 0.0)
        df['value'] = df['value'].apply(
            lambda x: float(x) if pd.notnull(x) and str(x).replace('.', '', 1).isdigit() else 0.0
        )

    # Apply your IPQ extraction function
    df['ipq'] = df['catalog_content'].apply(extract_ipq_from_text)

print(train[['sample_id', 'price']].head())


   sample_id  price
0      33127   4.89
1     198967  13.12
2     261251   1.97
3      55858  30.34
4     292686  66.49


In [8]:
print('Loading text model:', TEXT_MODEL)
text_model = SentenceTransformer(TEXT_MODEL)
text_model.to(DEVICE)

text_emb_train_path = os.path.join(ARTIFACTS, 'text_emb_train.npy')
text_emb_test_path  = os.path.join(ARTIFACTS, 'text_emb_test.npy')

if os.path.exists(text_emb_train_path) and os.path.exists(text_emb_test_path):
    print('Loading cached text embeddings...')
    text_emb_train = np.load(text_emb_train_path)
    text_emb_test  = np.load(text_emb_test_path)
else:
    print('Computing text embeddings...')
    text_emb_train = text_model.encode(train['clean_text'].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    text_emb_test  = text_model.encode(test['clean_text'].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    np.save(text_emb_train_path, text_emb_train)
    np.save(text_emb_test_path, text_emb_test)

print('Text embedding shapes:', text_emb_train.shape, text_emb_test.shape)


Loading text model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing text embeddings...


Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Text embedding shapes: (75000, 384) (75000, 384)


In [ ]:
img_emb_train_path = os.path.join(ARTIFACTS, 'img_emb_train.npy')
img_emb_test_path  = os.path.join(ARTIFACTS, 'img_emb_test.npy')
img_folder = os.path.join(ARTIFACTS, 'images')
os.makedirs(img_folder, exist_ok=True)

if os.path.exists(img_emb_train_path) and os.path.exists(img_emb_test_path):
    print('Loading cached img embeddings...')
    img_emb_train = np.load(img_emb_train_path)
    img_emb_test  = np.load(img_emb_test_path)
else:
    print('Building image model...')
    img_model = timm.create_model(IMAGE_MODEL, pretrained=True, num_classes=0, global_pool='avg')
    img_model.eval()
    img_model.to(DEVICE)
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    def img_emb_for_urls(urls, prefix):
        embs = []
        for i, url in enumerate(tqdm(urls, desc=prefix)):
            fname = os.path.join(img_folder, f"{prefix}_{i}.jpg")
            if url and (not os.path.exists(fname)):
                download_image(url, fname)
            if os.path.exists(fname):
                try:
                    img = Image.open(fname).convert('RGB')
                    x = transform(img).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        e = img_model(x).cpu().numpy().reshape(-1)
                except Exception:
                    e = np.zeros(getattr(img_model, 'num_features', 2048), dtype=float)
            else:
                e = np.zeros(getattr(img_model, 'num_features', 2048), dtype=float)
            embs.append(e)
        return np.vstack(embs)

    img_emb_train = img_emb_for_urls(train['image_link'].fillna('').tolist(), 'train')
    img_emb_test  = img_emb_for_urls(test['image_link'].fillna('').tolist(), 'test')
    np.save(img_emb_train_path, img_emb_train)
    np.save(img_emb_test_path, img_emb_test)

print('Image embedding shapes:', img_emb_train.shape, img_emb_test.shape)


Building image model...


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

train:  18%|█▊        | 13816/75000 [23:35<2:01:34,  8.39it/s]

In [ ]:
num_cols = ['num_words','num_chars','has_image','ipq','value']
X_num_train = train[num_cols].fillna(0).values
X_num_test  = test[num_cols].fillna(0).values

X_train = np.hstack([text_emb_train, img_emb_train, X_num_train])
X_test  = np.hstack([text_emb_test,  img_emb_test,  X_num_test])

y_train = np.log1p(train['price'].values)
print('Final feature shapes:', X_train.shape, X_test.shape)


In [ ]:
train['price_bin'] = pd.qcut(train['price'].clip(0.01), q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))
preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, train['price_bin'])):
    print('\nFold', fold+1)
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'learning_rate': 0.05,
        'num_leaves': 256,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'verbose': -1,
        'seed': 42
    }
    model = lgb.train(params, dtrain, num_boost_round=2000,
                      valid_sets=[dval], early_stopping_rounds=50, verbose_eval=100)
    oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    preds += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits
    model.save_model(os.path.join(ARTIFACTS, f'lgb_fold{fold+1}.txt'))


In [ ]:
def smape_np(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom[denom==0] = 1
    return np.mean(np.abs(y_pred - y_true) / denom) * 100

oof_price = np.expm1(oof)
cv_smape = smape_np(train['price'].values, oof_price)
print('\nCV SMAPE:', cv_smape)

final_preds = np.expm1(preds)
final_preds = np.clip(final_preds, 0.01, None)
submission = pd.DataFrame({'sample_id': test['sample_id'], 'price': final_preds})
submission_path = os.path.join(DATA_DIR, 'test_out.csv')
submission.to_csv(submission_path, index=False)
print('Saved submission to', submission_path)


In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception:
    print('If running locally, check:', submission_path)
